In [ ]:
# pdf_parser

import fitz

def extract_text(pdf_file):
    doc = fitz.open(stream=pdf_file.read(), filetype="pdf")

    text = ""

    for page in doc:
        text += page.get_text()

    return text

In [ ]:
# claim_extractor

from openai import OpenAI
import json

client = OpenAI()

def extract_claims(text):

    prompt = f"""
    Extract all factual claims.

    Focus on:
    - Statistics
    - Dates
    - Percentages
    - Financial figures
    - Technical metrics

    Return JSON array only.

    Text:
    {text[:15000]}
    """

    response = client.chat.completions.create(
        
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}]
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
# web_search.py

from tavily import TavilyClient
import os

client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)

def search_claim(claim):

    result = client.search(
        query=claim,
        search_depth="advanced",
        max_results=5
    )

    return result

In [ ]:
# verifier

from openai import OpenAI
import json

client = OpenAI()

def verify_claim(claim, evidence):

    prompt = f"""
    You are a professional fact checker.

    Claim:
    {claim}

    Evidence:
    {evidence}

    Classify:

    VERIFIED
    INACCURATE
    FALSE

    Return:

    {{
      "status":"",
      "correct_fact":"",
      "explanation":"",
      "confidence":0
    }}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}]
    )

    return json.loads(response.choices[0].message.content)